# 🔍 verificacion_pre_m01a.ipynb

**Verificación rápida antes de modificar `f6_m01a_shap_global.ipynb`**

## Qué hace

Comprueba 4 cosas, una por celda:

1. **Imports** — ¿están instaladas las librerías necesarias?
2. **Modelos** — ¿cargan los 3 .pkl (LightGBM, MLP, EBM)?
3. **Datos** — ¿se carga `X_test_prep.parquet` correctamente?
4. **SHAP rápido** — ¿LightGBM SHAP funciona en 30 segundos?

**No modifica nada. Solo verifica.**

Si alguna celda falla → te dice qué falta y lo arreglamos antes de tocar m01a.

In [1]:
# ── Celda 1: verificar imports y entorno ─────────────────────────────────────
import sys
from pathlib import Path

# ROOT detection robusto
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print(f'ROOT: {ROOT}')
print(f'Python: {sys.version.split()[0]}')
print()

# Verificar librerías necesarias
librerias_ok = True
for lib in ['joblib', 'numpy', 'pandas', 'shap', 'lightgbm', 'sklearn']:
    try:
        mod = __import__(lib)
        version = getattr(mod, '__version__', 'desconocida')
        print(f'  ✅ {lib}: {version}')
    except ImportError:
        print(f'  ❌ {lib}: NO instalado')
        librerias_ok = False

# Comprobar interpret (para EBM)
try:
    import interpret
    print(f'  ✅ interpret (EBM): {interpret.__version__}')
except ImportError:
    print(f'  ⚠️ interpret (EBM): NO instalado — necesario solo para EBM')

print()
if librerias_ok:
    print('✅ Todas las librerías esenciales están disponibles')
else:
    print('❌ Faltan librerías — instálalas antes de seguir')

ROOT: c:\FF\AU_UJI_v2
Python: 3.11.14

  ✅ joblib: 1.5.3
  ✅ numpy: 2.3.5
  ✅ pandas: 2.3.3
  ✅ shap: 0.51.0
  ✅ lightgbm: 4.6.0
  ✅ sklearn: 1.6.1
  ✅ interpret (EBM): 0.7.6

✅ Todas las librerías esenciales están disponibles


In [2]:
# ── Celda 2: verificar que los 3 .pkl cargan bien ────────────────────────────
import joblib
import json

DIR_MODELS = ROOT / 'data' / '05_modelado' / 'models'

# Leer ganador del JSON dinámicamente
RUTA_JSON = ROOT / 'data' / '06_evaluacion' / 'metricas_modelo.json'
with open(RUTA_JSON, encoding='utf-8') as f:
    meta_json = json.load(f)

nombre_ganador_pkl = meta_json['modelo_pkl']
print(f'📌 Modelo ganador (según JSON): {nombre_ganador_pkl}')
print()

# Modelos a cargar para SHAP comparativo (3 familias distintas)
modelos_check = {
    'GANADOR (Gradient Boosting)':  nombre_ganador_pkl,
    'MLP (Red Neuronal)':           'MLP__none.pkl',
    'EBM (Modelo Aditivo)':         'EBM__none.pkl',
}

modelos_cargados = {}
for etiqueta, pkl in modelos_check.items():
    ruta = DIR_MODELS / pkl
    if not ruta.exists():
        print(f'  ❌ {etiqueta}: NO existe {pkl}')
        continue
    try:
        modelo = joblib.load(ruta)
        # Si es Pipeline, extraer el estimador final
        if hasattr(modelo, 'named_steps'):
            estim = modelo.named_steps.get('model', modelo)
            tipo = type(estim).__name__
        else:
            tipo = type(modelo).__name__
        modelos_cargados[etiqueta] = modelo
        print(f'  ✅ {etiqueta}: {pkl}  →  estimador: {tipo}')
    except Exception as e:
        print(f'  ❌ {etiqueta}: error al cargar — {e}')

print()
print(f'Modelos cargados: {len(modelos_cargados)}/3')

📌 Modelo ganador (según JSON): LightGBM__none.pkl

  ✅ GANADOR (Gradient Boosting): LightGBM__none.pkl  →  estimador: LGBMClassifier
  ✅ MLP (Red Neuronal): MLP__none.pkl  →  estimador: MLPClassifier
  ✅ EBM (Modelo Aditivo): EBM__none.pkl  →  estimador: ExplainableBoostingClassifier

Modelos cargados: 3/3


In [3]:
# ── Celda 3: verificar X_test_prep ───────────────────────────────────────────
import pandas as pd

RUTA_X = ROOT / 'data' / '05_modelado' / 'X_test_prep.parquet'
assert RUTA_X.exists(), f'❌ No encontrado: {RUTA_X}'

X_test_prep = pd.read_parquet(RUTA_X)
print(f'✅ X_test_prep: {X_test_prep.shape}')
print(f'   Features: {X_test_prep.shape[1]}')
print()

# Verificar predict_proba con cada modelo
print('Probando predict_proba con cada modelo:')
for etiqueta, modelo in modelos_cargados.items():
    try:
        prob = modelo.predict_proba(X_test_prep.head(5))[:, 1]
        print(f'  ✅ {etiqueta}: probs = {[f"{p:.3f}" for p in prob]}')
    except Exception as e:
        print(f'  ❌ {etiqueta}: error — {str(e)[:80]}')

✅ X_test_prep: (6725, 27)
   Features: 27

Probando predict_proba con cada modelo:
  ✅ GANADOR (Gradient Boosting): probs = ['0.017', '0.081', '0.005', '0.019', '0.027']
  ✅ MLP (Red Neuronal): probs = ['0.002', '0.029', '0.000', '0.010', '0.008']
  ✅ EBM (Modelo Aditivo): probs = ['0.023', '0.309', '0.002', '0.027', '0.029']


In [4]:
# ── Celda 4: SHAP rápido sobre LightGBM (30 segundos) ───────────────────────
# Solo para verificar que TreeExplainer funciona con el ganador.
# NO calculamos los 6.725 — solo 100 alumnos para test rápido.
import shap
import time

modelo_ganador = modelos_cargados['GANADOR (Gradient Boosting)']

# Extraer el estimador final si es Pipeline
if hasattr(modelo_ganador, 'named_steps'):
    ganador_raw = modelo_ganador.named_steps['model']
else:
    ganador_raw = modelo_ganador

print(f'Estimador a explicar: {type(ganador_raw).__name__}')
print(f'Muestra de prueba: 100 alumnos')
print()

X_muestra = X_test_prep.head(100)

t0 = time.time()
explainer = shap.TreeExplainer(ganador_raw)
shap_values = explainer(X_muestra)
t1 = time.time()

print(f'✅ SHAP calculado en {t1-t0:.1f}s')
print(f'   Shape de shap_values: {shap_values.values.shape}')
print(f'   Esperado: (100, {X_test_prep.shape[1]})')
print()

# Top 5 features por importancia
import numpy as np
importancias = np.abs(shap_values.values).mean(axis=0)
top5_idx = np.argsort(importancias)[::-1][:5]
print('Top 5 features (por importancia SHAP media):')
for i, idx in enumerate(top5_idx, 1):
    print(f'  {i}. {X_test_prep.columns[idx]:30}  →  {importancias[idx]:.4f}')

Estimador a explicar: LGBMClassifier
Muestra de prueba: 100 alumnos

✅ SHAP calculado en 0.4s
   Shape de shap_values: (100, 27)
   Esperado: (100, 27)

Top 5 features (por importancia SHAP media):
  1. cred_superados_anio_1er         →  0.7467
  2. n_anios_beca                    →  0.6283
  3. n_anios_trabajando              →  0.6211
  4. cred_repetidos                  →  0.4650
  5. anios_sin_beca                  →  0.4239


## ✅ Si las 4 celdas dan verde

Significa que:
- ✅ Las librerías están instaladas
- ✅ Los 3 modelos cargan bien
- ✅ Los datos se cargan bien
- ✅ SHAP funciona con LightGBM

→ **Listos para modificar m01a con seguridad.**

## ⚠️ Si alguna falla

Pega el error en el chat y lo arreglamos antes de tocar m01a.